In [ ]:
!pip install -q pydantic aiohttp pandas openpyxl tenacity nest_asyncio

In [ ]:

# ==========================================
# CELDA 2: EL CEREBRO DE VALIDACIÓN MATEMÁTICO
# ==========================================
import nest_asyncio
nest_asyncio.apply()
import os
import re
import asyncio
import aiohttp
import hashlib
from difflib import SequenceMatcher
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any, Union
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type




class PublicationRecord(BaseModel):
    id_registro: str = Field(..., description="ID of the record in the matrix")
    tipo_publicacion: Optional[str] = Field(None, description="Type of publication (e.g., ARTICULO, LIBRO)")
    codigo_doi: Optional[str] = Field(None, description="Registered DOI code")
    codigo_issn: Optional[str] = Field(None, description="Registered ISSN code")
    titulo: Optional[str] = Field(None, description="Registered title")
    autores: Optional[str] = Field(None, description="Registered authors")
    anio_publicacion: Optional[str] = Field(None, description="Registered publication year")
    revista_editorial: Optional[str] = Field(None, description="Registered journal or publisher")

class ValidationResult(BaseModel):
    codigo: str = Field(..., description="DOI or ISSN code validated")
    publicacion_encontrada: str = Field(..., description="Details of the publication found in official sources")
    datos_registrados: str = Field(..., description="Details of the registered data")
    coincide: str = Field(..., description="Does it match? 'Sí' or 'No'")
    observaciones: str = Field(..., description="Inconsistencies, missing info, format errors, etc.")
    similitud: float = Field(0.0, description="Percentage of similarity (0.0 to 100.0)")
    
class AIValidationReport(BaseModel):
    resultados: List[ValidationResult]







class MetadataFetcher:
    """
    Async Client for fetching metadata from public APIs (Crossref/OpenAlex) 
    with Exponential Backoff retries, persistent sessions and Polite Pool access.
    """
    def __init__(self):
        self.session: Optional[aiohttp.ClientSession] = None

    async def get_session(self) -> aiohttp.ClientSession:
        if self.session is None or self.session.closed:
            headers = {
                "User-Agent": "PublicationValidatorBot/1.0 (mailto:soporte@tudominio.com)"
            }
            timeout = aiohttp.ClientTimeout(total=15)
            self.session = aiohttp.ClientSession(timeout=timeout, headers=headers)
        return self.session

    async def close(self):
        if self.session and not self.session.closed:
            await self.session.close()

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_exception_type((aiohttp.ClientError, asyncio.TimeoutError))
    )
    async def fetch_doi_metadata(self, doi: str) -> Optional[Dict[str, Any]]:
        """
        Fetches metadata for a given DOI asynchronusly.
        """
        url = f"https://api.crossref.org/works/{doi}"
        session = await self.get_session()
        
        async with session.get(url) as response:
            if response.status == 200:
                data = await response.json()
                message = data.get("message", {})
                
                title = message.get("title", [""])[0] if message.get("title") else ""
                authors_list = message.get("author", [])
                
                # Para evitar exceder el límite de tokens de Groq (6000 TPM) con papers que tienen
                # miles de autores (ej. papers de física), extraemos máximo los primeros 20 autores
                # y truncamos la cadena final a 800 caracteres.
                authors_str_list = [f"{a.get('given', '')} {a.get('family', '')}".strip() for a in authors_list[:30]]
                authors = ", ".join(authors_str_list)
                if len(authors_list) > 30:
                    authors += " [y otros...]"
                    
                # Truncar título también por seguridad
                if len(title) > 800:
                    title = title[:800] + "..."
                
                issued = message.get("issued", {}).get("date-parts", [[None]])[0][0]
                year = str(issued) if issued else ""
                
                container_title = message.get("container-title", [""])[0] if message.get("container-title") else ""
                publisher = message.get("publisher", "")
                journal_or_publisher = container_title if container_title else publisher
                if len(journal_or_publisher) > 500:
                    journal_or_publisher = journal_or_publisher[:500] + "..."
                
                pub_type = message.get("type", "")
                
                return {
                    "titulo": title,
                    "autores": authors,
                    "anio_publicacion": year,
                    "revista_editorial": journal_or_publisher,
                    "tipo_publicacion": pub_type
                }
            elif response.status == 404:
                return None
            else:
                response.raise_for_status()

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_exception_type((aiohttp.ClientError, asyncio.TimeoutError))
    )
    async def fetch_issn_metadata(self, issn: str) -> Optional[Dict[str, Any]]:
        """
        Fetches metadata for a given ISSN asynchronously.
        """
        url = f"https://api.crossref.org/journals/{issn}"
        session = await self.get_session()
        
        async with session.get(url) as response:
            if response.status == 200:
                data = await response.json()
                message = data.get("message", {})
                
                title = message.get("title", "")
                publisher = message.get("publisher", "")
                
                return {
                    "titulo": title,
                    "editorial": publisher,
                    "tipo_publicacion": "journal"
                }
            elif response.status == 404:
                # Fallback to OpenAlex
                openalex_url = f"https://api.openalex.org/sources/issn:{issn}"
                async with session.get(openalex_url) as oa_response:
                    if oa_response.status == 200:
                        data = await oa_response.json()
                        title = data.get("display_name", "")
                        publisher = data.get("host_organization_name", "")
                        country = data.get("country_code", "")
                        return {
                            "titulo": title,
                            "editorial": publisher,
                            "pais": country,
                            "tipo_publicacion": data.get("type", "")
                        }
                    elif oa_response.status == 404:
                        return None
                    else:
                        oa_response.raise_for_status()
            else:
                response.raise_for_status()










class SimilarityEngine:
    """
    Motor matemático local para comparar cadenas de texto ignorando mayúsculas, signos y acentos.
    """
    def _normalize_text(self, text: str) -> str:
        if not text:
            return ""
        # Convertir a minúsculas
        text = str(text).lower()
        # Eliminar tildes básicas (simplificado para no depender de librerías extra como unidecode)
        replacements = (("á", "a"), ("é", "e"), ("í", "i"), ("ó", "o"), ("ú", "u"), ("ü", "u"), ("ñ", "n"))
        for a, b in replacements:
            text = text.replace(a, b)
        # Eliminar todo lo que no sea letra o número
        text = re.sub(r'[^\w\s]', '', text)
        # Eliminar espacios dobles
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def compare_titles(self, title1: str, title2: str, threshold: float = 0.85) -> dict:
        norm1 = self._normalize_text(title1)
        norm2 = self._normalize_text(title2)
        
        if not norm1 or not norm2:
            return {"coincide": "No", "similitud": 0.0, "observaciones": "Uno de los títulos está en blanco."}
            
        ratio = SequenceMatcher(None, norm1, norm2).ratio()
        similitud_porcentaje = round(ratio * 100, 2)
        
        coincide = "Sí" if ratio >= threshold else "No"
        
        if coincide == "Sí":
            obs = f"Aprobado: El título coincide matemáticamente al {similitud_porcentaje}%."
        else:
            obs = f"Rechazado: El título registrado es muy distinto al oficial (Similitud: {similitud_porcentaje}%)."
            
        return {"coincide": coincide, "similitud": similitud_porcentaje, "observaciones": obs}


class PublicationValidator:
    """
    Main Orchestrator for validating DOIs and ISSNs using Mathematical Similarity (No AI).
    """
    def __init__(self, fetcher: MetadataFetcher = None, threshold: float = 0.85):
        self.fetcher = fetcher or MetadataFetcher()
        self.engine = SimilarityEngine()
        self.threshold = threshold
        self._cache: Dict[str, ValidationResult] = {}
        
    def _clean_code(self, code: Union[str, None]) -> str:
        return code.strip() if code else ""

    def _limpiar_codigo(self, raw: str) -> str:
        if not raw: return ""
        cleaned = str(raw).strip()
        if "doi.org/" in cleaned:
            cleaned = cleaned.split("doi.org/")[-1]
        
        cleaned = re.sub(r'(?i)^doi:\s*', '', cleaned)
        cleaned = re.sub(r'(?i)^issn:\s*', '', cleaned)
        
        if ":" in cleaned and not cleaned.startswith("10."):
            partes = cleaned.split(":", 1)
            if len(partes) > 1 and (partes[1].strip().startswith("10.") or re.match(r'^\d', partes[1].strip())):
                cleaned = partes[1].strip()
                
        cleaned = re.sub(r'[^a-zA-Z0-9.\-/:_;()]', '', cleaned)
        return cleaned

    def _is_valid_doi_format(self, doi: str) -> bool:
        if not doi:
            return False
        return bool(re.match(r'^10\.\d{4,9}/[-._;()/:A-Z0-9]+$', doi, re.IGNORECASE))
        
    def _is_valid_issn_format(self, issn: str) -> bool:
        if not issn:
            return False
        return bool(re.match(r'^\d{4}-?\d{3}[\dxX]$', issn, re.IGNORECASE))

    async def validate_record(self, record: Union[Dict[str, Any], PublicationRecord]) -> ValidationResult:
        if isinstance(record, dict):
            record = PublicationRecord(**record)
            
        doi_limpio = self._clean_code(record.codigo_doi)
        issn_limpio = self._clean_code(record.codigo_issn)
        
        record_str = f"{record.titulo}_{doi_limpio}_{issn_limpio}"
        record_hash = hashlib.md5(record_str.encode('utf-8')).hexdigest()
        cache_key = f"code_{record_hash}"
        
        if cache_key in self._cache:
            return self._cache[cache_key]
        
        if doi_limpio:
            if self._is_valid_doi_format(doi_limpio):
                record.codigo_doi = doi_limpio
                try:
                    official_data = await self.fetcher.fetch_doi_metadata(doi_limpio)
                except Exception:
                    official_data = None
                    
                if not official_data:
                    result = ValidationResult(
                        codigo=doi_limpio,
                        publicacion_encontrada="No se encontró en la Base de Datos",
                        datos_registrados=record.titulo or "Desconocido",
                        coincide="No",
                        observaciones="El DOI es inválido o no existe en Crossref.",
                        similitud=0.0
                    )
                else:
                    official_title = official_data.get("titulo", "")
                    user_title = record.titulo or ""
                    
                    comp = self.engine.compare_titles(user_title, official_title, self.threshold)
                    
                    result = ValidationResult(
                        codigo=doi_limpio,
                        publicacion_encontrada=official_title,
                        datos_registrados=user_title,
                        coincide=comp["coincide"],
                        observaciones=comp["observaciones"],
                        similitud=comp["similitud"]
                    )
                    
                self._cache[cache_key] = result
                return result
            else:
                return ValidationResult(
                    codigo=doi_limpio,
                    publicacion_encontrada="No se consultó",
                    datos_registrados=record.titulo or "Desconocido",
                    coincide="No",
                    observaciones="Error de formato: El DOI es incorrecto.",
                    similitud=0.0
                )
                
        elif issn_limpio:
            if self._is_valid_issn_format(issn_limpio):
                record.codigo_issn = issn_limpio
                try:
                    official_data = await self.fetcher.fetch_issn_metadata(issn_limpio)
                except Exception:
                    official_data = None
                    
                if not official_data:
                    result = ValidationResult(
                        codigo=issn_limpio,
                        publicacion_encontrada="No se encontró en la Base de Datos",
                        datos_registrados=record.titulo or "Desconocido",
                        coincide="No",
                        observaciones="El ISSN es inválido o no existe en los catálogos.",
                        similitud=0.0
                    )
                else:
                    official_title = official_data.get("titulo", "")
                    user_title = record.titulo or ""
                    
                    comp = self.engine.compare_titles(user_title, official_title, self.threshold)
                    
                    result = ValidationResult(
                        codigo=issn_limpio,
                        publicacion_encontrada=official_title,
                        datos_registrados=user_title,
                        coincide=comp["coincide"],
                        observaciones=comp["observaciones"],
                        similitud=comp["similitud"]
                    )
                    
                self._cache[cache_key] = result
                return result
            else:
                return ValidationResult(
                    codigo=issn_limpio,
                    publicacion_encontrada="No se consultó",
                    datos_registrados=record.titulo or "Desconocido",
                    coincide="No",
                    observaciones="Error de formato: El ISSN es incorrecto.",
                    similitud=0.0
                )
                
        else:
            return ValidationResult(
                codigo="N/A",
                publicacion_encontrada="No se consultó",
                datos_registrados=record.titulo or "Desconocido",
                coincide="No",
                observaciones="No se proporcionó ningún DOI o ISSN.",
                similitud=0.0
            )

    async def validate_batch(self, records: List[Union[Dict[str, Any], PublicationRecord]]) -> AIValidationReport:
        # Ya no requerimos un semáforo artificial (ai_sem) porque ya no usamos Groq.
        # Enviaremos las peticiones concurrentemente de a 20 a la vez para cuidar OpenAlex/Crossref.
        sem = asyncio.Semaphore(20)
        
        async def wrap_validate(rec):
            async with sem:
                return await self.validate_record(rec)
                
        tasks = [wrap_validate(rec) for rec in records]
        results = await asyncio.gather(*tasks)
        
        if hasattr(self.fetcher, 'close'):
            await self.fetcher.close()
            
        return AIValidationReport(resultados=list(results))



In [ ]:

# ==========================================
# CELDA 3: INTERFAZ DE USUARIO Y EJECUCIÓN
# ==========================================
from google.colab import files
import pandas as pd
import io

print("=== VALIDADOR MATEMÁTICO DE DOI E ISSN (GOOGLE COLAB) ===")
print("\n📂 Sube tu archivo Excel (debe tener las columnas TIPO, CODIGO, TITULO)")
uploaded = files.upload()

if not uploaded:
    print("❌ No se subió ningún archivo.")
else:
    filename = list(uploaded.keys())[0]
    df = pd.read_excel(io.BytesIO(uploaded[filename]))
    print(f"✅ Archivo cargado: {len(df)} registros encontrados.")
    
    registros_a_procesar = []
    for idx, row in df.iterrows():
        tipo = str(row.get("TIPO", "")).strip().upper()
        codigo = str(row.get("CODIGO", "")).strip()
        titulo = str(row.get("TITULO", "")).strip() if pd.notna(row.get("TITULO", "")) else ""
        
        if tipo not in ["DOI", "ISSN"]:
            tipo = "DOI" if "10." in codigo else "ISSN"
            
        registro = {
            "id_registro": f"Fila_{idx+2}",
            "tipo_publicacion": "ARTICULO" if tipo == "DOI" else "REVISTA",
            "titulo": titulo if titulo else "Desconocido",
            "autores": "",
            "anio_publicacion": "",
            "revista_editorial": ""
        }
        
        if tipo == "DOI":
            registro["codigo_doi"] = codigo
            registro["codigo_issn"] = ""
        else:
            registro["codigo_doi"] = ""
            registro["codigo_issn"] = codigo
            
        registros_a_procesar.append(registro)
        
    async def procesar_lote():
        validador = PublicationValidator(threshold=0.85)
        print("\n🚀 Iniciando validación rápida...")
        report = await validador.validate_batch(registros_a_procesar)
        return report.resultados
            
    resultados_list = asyncio.run(procesar_lote())
    
    print("\n💾 Generando archivo de resultados...")
    df_out = df.copy()
    df_out["%_SIMILITUD"] = [res.similitud for res in resultados_list]
    df_out["COINCIDE_MATH"] = [res.coincide for res in resultados_list]
    df_out["OBSERVACIONES"] = [res.observaciones for res in resultados_list]
    df_out["TITULO_OFICIAL"] = [res.publicacion_encontrada for res in resultados_list]
    
    output_filename = f"Resultados_Math_Colab_{filename}"
    df_out.to_excel(output_filename, index=False)
    
    print(f"🎉 ¡Proceso finalizado! Descargando {output_filename}...")
    files.download(output_filename)
